# Project 1. Data Preprocessing

# Credit Reliability Study of Bank Customers

**Goal:** Practice data preprocessing: handle missing values, work with data types, extract lemmas and categorize data. Study how family status, number of children, income level and the purpose of the loan affect the repayment of a loan on time.

**Plan:**
- 1. General information about the data
- 2. Data preprocessing
    - 2.1 Filling in the missing values
    - 2.2 Changing the data types
    - 2.3 Removing duplicates
    - 2.4 Lemmatization of the loan purposes
    - 2.5 Data categorization
- 3. Data analysis
    - 3.1 Dependence between having children and repaying a loan on time
    - 3.2 Dependence between family status and repaying a loan on time
    - 3.3 Dependence between income level and repaying a loan on time
    - 3.4 Influence of the loan purpose on repaying it on time
- 4. Overall conclusion

**Data description:**
The data is a table with statistics about the solvency of bank customers:
- `children` - number of children in the family
- `days_employed` - total work experience in days
- `dob_years` - customer's age in years
- `education` - customer's education level
- `education_id` - education level identifier
- `family_status` - family status
- `family_status_id` - family status identifier
- `gender` - customer's gender
- `income_type` - employment type
- `debt` - whether the customer had a debt on loan repayment
- `total_income` - monthly income
- `purpose` - purpose of the loan

> The text values in the columns `education`, `family_status`, `income_type` and `purpose` are in Russian, as in the original dataset.

## Step 1. General Information About the Data

In [1]:
# import pandas to work with tables and display to print tables
import pandas as pd
from IPython.display import display
# import the morphological analyzer from pymorphy3 (Russian lemmatization)
from pymorphy3 import MorphAnalyzer

In [2]:
# read the data from the .csv file with read_csv()
customer_solvency = pd.read_csv('data.csv')
# print the first 10 rows of the table with head()
display(customer_solvency.head(10))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


In [3]:
# study the table structure with info(): data types, number of rows and columns,
# missing values.
customer_solvency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


### Conclusion

The data is loaded successfully and stored correctly in a DataFrame. The table has 12 columns and 21525 records. There are missing values, which are studied below.

## Step 2. Data Preprocessing

### 2.1 Filling in the Missing Values

The table has 21525 rows and 12 columns. Values are missing in `days_employed` (total work experience in days) and `total_income` (monthly income), and the number of missing values is the same in both columns. There may be a pattern.

In [4]:
# save in missed_days_employed all rows with a missing 'days_employed'
missed_days_employed = customer_solvency.loc[customer_solvency['days_employed'].isnull()]
print('Number of rows with missing work experience:', len(missed_days_employed))
# check that 'total_income' is also missing in all of these rows
display(missed_days_employed.loc[missed_days_employed['total_income'].isnull() == False])

Number of rows with missing work experience: 2174


,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose


About 10% of the rows have missing values. They cannot simply be removed without distorting the statistics.

#### Incorrect data in the column `days_employed`:

The values in `days_employed` and `total_income` are missing in the same rows. A system failure may have occurred when the data was exported or recorded. The negative values in `days_employed` also point to this.
Errors during data entry are also likely: a total work experience of 340266 days looks implausible for a customer aged 53. A person's work experience cannot be more than 30000 days (a rough estimate: 75-80 years of work). Find the number of records that exceed this value.

In [5]:
# select the records with 'days_employed' > 30000 with loc[] and a logical condition,
# and count them with len()
wrong_days_employed = len(customer_solvency.loc[customer_solvency['days_employed'] > 30000])
print('Number of rows with incorrect work experience:', wrong_days_employed)
# in the same way find the number of rows with negative work experience.
neg_days_employed = len(customer_solvency.loc[customer_solvency['days_employed'] < 0])
print('Number of rows with negative work experience:', neg_days_employed)

Number of rows with incorrect work experience: 3445
Number of rows with negative work experience: 15906


- Almost all values in `days_employed` are incorrect or missing: about 90% are negative or implausibly large, and about 10% are missing. The reasons are unknown, so changing the incorrect data could only make the situation worse.
- The values may have been entered in hours (instead of days), which would explain the high values. A system failure could also have changed the data format (the reference point of the days could have changed, which led to the negative values).
- In any case the problem needs further investigation together with the bank employees.
- At this stage the missing values in `days_employed` are replaced with zeros for clarity.

In [6]:
# find all rows with a missing value in 'days_employed' with isnull() and replace
# them with 0
customer_solvency['days_employed'] = customer_solvency['days_employed'].fillna(0)
display(customer_solvency.loc[customer_solvency['days_employed'] == 0].head(10))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
12,0,0.0,65,среднее,1,гражданский брак,1,M,пенсионер,0,NaN,сыграть свадьбу
26,0,0.0,41,среднее,1,женат / замужем,0,M,госслужащий,0,NaN,образование
29,0,0.0,63,среднее,1,Не женат / не замужем,4,F,пенсионер,0,NaN,строительство жилой недвижимости
41,0,0.0,50,среднее,1,женат / замужем,0,F,госслужащий,0,NaN,сделка с подержанным автомобилем
55,0,0.0,54,среднее,1,гражданский брак,1,F,пенсионер,1,NaN,сыграть свадьбу
65,0,0.0,21,среднее,1,Не женат / не замужем,4,M,компаньон,0,NaN,операции с коммерческой недвижимостью
67,0,0.0,52,высшее,0,женат / замужем,0,F,пенсионер,0,NaN,покупка жилья для семьи
72,1,0.0,32,высшее,0,женат / замужем,0,M,госслужащий,0,NaN,операции с коммерческой недвижимостью
82,2,0.0,50,высшее,0,женат / замужем,0,F,сотрудник,0,NaN,жилье
83,0,0.0,52,среднее,1,женат / замужем,0,M,сотрудник,0,NaN,жилье


- Look at the column `total_income`.
Group the data by education level and print the income statistics of each group (minimum, maximum, mean and median).

In [7]:
# group the values by 'education_id' with groupby().
# apply statistical functions to the column 'total_income' with agg()
customer_solvency_stat = customer_solvency.groupby('education_id').agg({'total_income':['min','max','mean','median']})
display(customer_solvency_stat)

total_income                                            
                       min           max           mean         median
education_id                                                          
0             32178.213678  2.265604e+06  207142.515219  175340.818855
1             20667.263793  1.726276e+06  153715.643971  136478.643244
2             34466.133539  9.584346e+05  181534.022774  160115.398644
3             25308.586849  4.900673e+05  132155.513626  117137.352825
4             98752.495442  2.684112e+05  174750.155792  157259.898555

The minimum and maximum values of the monthly income differ from the mean by a factor of about 10. The means also differ strongly from the medians. In this case the missing values are filled with the medians.

In [8]:
# use fillna() to replace the missing values
# in the argument of the method group the data by 'education_id' and calculate the median
# 'total_income' of each group
customer_solvency['total_income'] = customer_solvency['total_income'].fillna(customer_solvency.groupby('education_id')['total_income'].transform('median'))

- Look at the column `gender`:

In [9]:
# find the number of occurrences of each value with value_counts()
display(customer_solvency['gender'].value_counts())
display(customer_solvency.loc[customer_solvency['gender'] == 'XNA'])

F      14236
M       7288
XNA        1
Name: gender, dtype: int64

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
10701,0,-2358.600502,24,неоконченное высшее,2,гражданский брак,1,XNA,компаньон,0,203905.157261,покупка недвижимости


The incorrect value `XNA` occurs once. The other values of this record do not allow determining the gender of the customer. One record can be removed without affecting the statistics.

In [10]:
customer_solvency = customer_solvency.loc[customer_solvency['gender'] != 'XNA']

### Conclusion

About 10% of the records have missing values. In the data that is not used in the further study the missing values were replaced with zeros; in `total_income` they were replaced with the median values of the groups with the same customer education level. Single rows with incorrect data were removed.
In `days_employed` almost all values are missing or incorrect (about 90% are negative or implausibly large, about 10% are missing). The reasons need further investigation together with the bank employees.

### 2.2 Changing the Data Types

- The number of working days can only be an integer. The monthly income is also made an integer for clarity.
- Convert the data types in `days_employed` and `total_income` from `float64` to the integer `int64`.
- The conversion is done with `astype()`. It changes the type of real numbers and raises an error if the data was entered incorrectly (for example, strings instead of real numbers).

In [11]:
# convert the data types with astype().
# use try-except to catch incorrect data types
try:
    customer_solvency['days_employed'] = customer_solvency['days_employed'].astype('int64')
    customer_solvency['total_income'] = customer_solvency['total_income'].astype('int64')
except:
    print('Incorrect data type. Please make sure that the data is stored as real numbers')

### Conclusion

The real numbers are successfully converted to the integer type `int64` with `astype()`.

### 2.3 Removing Duplicates

- To search for duplicates, first convert all string columns to lower case.
- Then count the duplicated rows and print them for analysis.

In [12]:
# convert to lower case with str.lower()
# and write the converted data back to the table
customer_solvency['education'] = customer_solvency['education'].str.lower()
customer_solvency['family_status'] = customer_solvency['family_status'].str.lower()

In [13]:
print('Number of duplicated rows: ', customer_solvency.duplicated().sum())

Number of duplicated rows:  71


In [14]:
display(customer_solvency.loc[customer_solvency.duplicated()].head(10))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
2849,0,0,41,среднее,1,женат / замужем,0,F,сотрудник,0,136478,покупка жилья для семьи
3290,0,0,58,среднее,1,гражданский брак,1,F,пенсионер,0,136478,сыграть свадьбу
4182,1,0,34,высшее,0,гражданский брак,1,F,сотрудник,0,175340,свадьба
4851,0,0,60,среднее,1,гражданский брак,1,F,пенсионер,0,136478,свадьба
5557,0,0,58,среднее,1,гражданский брак,1,F,пенсионер,0,136478,сыграть свадьбу
6312,0,0,30,среднее,1,женат / замужем,0,M,сотрудник,0,136478,строительство жилой недвижимости
7808,0,0,57,среднее,1,гражданский брак,1,F,пенсионер,0,136478,на проведение свадьбы
7921,0,0,64,высшее,0,гражданский брак,1,F,пенсионер,0,175340,на проведение свадьбы
7938,0,0,71,среднее,1,гражданский брак,1,F,пенсионер,0,136478,на проведение свадьбы
8583,0,0,58,высшее,0,не женат / не замужем,4,F,пенсионер,0,175340,дополнительное образование


- All duplicated rows are rows in which missing values were processed earlier.
Since the table has no unique customer identifiers, it is impossible to say for sure whether the data is duplicated or the information about different customers coincided by chance.
- Possible reasons for the duplicates: the customer data was entered or saved in the database twice, or a system failure occurred while the database was copied or exported.
- The duplicated rows are removed for educational purposes with `drop_duplicates()`, and the indices are restored after the removal with `reset_index()`.

In [15]:
customer_solvency = customer_solvency.drop_duplicates().reset_index(drop = True)
print('Number of rows after removing the duplicates:', len(customer_solvency))

Number of rows after removing the duplicates: 21453


### Conclusion

71 duplicates were found and removed successfully.
Possible reasons for the duplicates: the customer data was entered or saved in the database twice, or a system failure occurred while the database was copied or exported.

### 2.4 Lemmatization of the Loan Purposes

Print all possible loan purposes for analysis with `value_counts()`.

In [16]:
# the table purposes stores the original loan purposes and the categories created after the lemmatization
purposes = pd.DataFrame(columns = ['purpose', 'purpose_category'])
# find all purposes in the table with unique() and write them to the new table
purposes['purpose'] = customer_solvency['purpose'].unique()
display(purposes)

,purpose,purpose_category
0,покупка жилья,NaN
1,приобретение автомобиля,NaN
2,дополнительное образование,NaN
3,сыграть свадьбу,NaN
4,операции с жильем,NaN
5,образование,NaN
6,на проведение свадьбы,NaN
7,покупка жилья для семьи,NaN
8,покупка недвижимости,NaN
9,покупка коммерческой недвижимости,NaN


The lemmatization is done with the library `pymorphy3`. Each string of the column `purpose` is converted to a list of lemmas, and the purposes are split into 6 groups:
**1) wedding:** several descriptions of purposes describe a loan for holding a wedding. The amount and the term of such a loan can be relatively small and differ strongly from loans for other purposes, so the statistics of debts may differ. For this reason a separate category is created.
**2) car:** this category contains loans for different operations with a car (purchase, exchange, other deals). As with the wedding, the amount and the term can be relatively small. The obligation to repay the debt can lie on a family (for a family car) or on one person, so the statistics of debts may differ. A separate group is created for this too.
**3) housing/real estate:** the table contains many purposes related to real estate. Some of them relate to private housing, others to commercial real estate, and private persons and legal entities can differ greatly. The table makes it hard to separate these two subcategories, so bank specialists would have to be asked for a more accurate split of purposes and customers. However, real estate loans mostly involve large amounts and long terms, and both private persons and legal entities treat such loans seriously. So the statistics of debts may be similar across the described subcategories and differ from the other groups. All real estate loans are put into one category.
**4) education:** an education loan can be taken by adult customers with a stable income and by students without a stable income who received an education grant. Such loans can be taken for a relatively long term (the study period), and the start of the repayment can be postponed for several years until graduation. Because of these special conditions a separate group is created.
**5) repair:** on the one hand this purpose correlates with 'housing/real estate', but the amount and the term of the loan are usually much smaller, so the debt results may differ. Therefore 'repair' is a separate category.
**6) other:** for possible mismatches with the main purposes.

In [17]:
# initialize the analyzer and save it in the variable m
m = MorphAnalyzer()
def lemm_categorize(purpose):
    """
    input: purpose (object) - string that describes the purpose of the loan
    output: (object) - string, name of the category of the loan purpose
    The function takes the purpose of a loan, lemmatizes it and determines
    the category of the purpose by the lemmas it contains
    (the lemmas are Russian words, because the purposes in the dataset are in Russian):
    
    Lemma:                  Category:
    свадьба (wedding)       wedding
    ремонт (repair)         repair
    автомобиль (car)        car
    образование (education) education
    жильё (housing)         housing/real estate
    недвижимость (real est) housing/real estate
    
    If the lemmas match none of the categories, the category 'other' is assigned
    """
    lemmas = [m.parse(word)[0].normal_form for word in purpose.split()]
    if 'свадьба' in lemmas:
        return 'wedding'
    elif 'ремонт' in lemmas:
        return 'repair'
    elif 'автомобиль' in lemmas:
        return 'car'
    elif 'образование' in lemmas:
        return 'education'
    elif 'жильё' in lemmas or 'недвижимость' in lemmas:
        return 'housing/real estate'
    else:
        return 'other'

In [18]:
# call the function lemm_categorize for each purpose in the table purposes with apply().
# the categories returned by the function are written to the column 'purpose_category'
purposes['purpose_category'] = purposes['purpose'].apply(lemm_categorize)

In [19]:
display(purposes.head(10))

,purpose,purpose_category
0,покупка жилья,housing/real estate
1,приобретение автомобиля,car
2,дополнительное образование,education
3,сыграть свадьбу,wedding
4,операции с жильем,housing/real estate
5,образование,education
6,на проведение свадьбы,wedding
7,покупка жилья для семьи,housing/real estate
8,покупка недвижимости,housing/real estate
9,покупка коммерческой недвижимости,housing/real estate


The categories are assigned to each loan purpose successfully. Add `purpose_category` to the main table `customer_solvency` with `merge()` below.

In [20]:
customer_solvency = customer_solvency.merge(purposes, on='purpose', how='left')
display(customer_solvency.head(10))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,purpose_category
0,1,-8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,housing/real estate
1,1,-4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,car
2,0,-5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,housing/real estate
3,3,-4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,education
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,wedding
5,0,-926,27,высшее,0,гражданский брак,1,M,компаньон,0,255763,покупка жилья,housing/real estate
6,0,-2879,43,высшее,0,женат / замужем,0,F,компаньон,0,240525,операции с жильем,housing/real estate
7,0,-152,50,среднее,1,женат / замужем,0,M,сотрудник,0,135823,образование,education
8,2,-6929,35,высшее,0,гражданский брак,1,F,сотрудник,0,95856,на проведение свадьбы,wedding
9,0,-2188,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425,покупка жилья для семьи,housing/real estate


### Conclusion

Among the loan purposes 5 main groups are selected: `wedding`, `repair`, `car`, `education`, `housing/real estate`.
For the purposes that do not fit any category the group `other` is created (for possible new purposes when more data is accumulated).
The purposes were lemmatized with the library `pymorphy3`.

### 2.5 Data Categorization

1. In the previous step a dictionary of loan purposes was created. Now the column `purpose` in the main table is replaced with the identifier `purpose_id`, and the description of each purpose is stored only in the dictionary. This way the main table takes less space and becomes clearer.
2. The family status `family_status` and the education level `education` are also removed from the main table, leaving only the identifiers `education_id` and `family_status_id`. The removed data is stored in the dictionaries `education_dict` and `family_status_dict`. The names for reports can be taken from the dictionaries later.
3. Categorize the number of children in the family:
    - 0: No children
    - 1: One child
    - 2: Two children
    - 3: Three or more children
    - 4: Status unknown (value -1 in the column `children`)

   This makes it possible to study the influence of having children on the credit history of customers.
4. Categorize the monthly income. Split the customers into 6 equal categories by income level (about 3500 customers in each). Equal sample sizes in each category make the statistical results of the study more reliable.
   This makes it possible to group the customers by income level and to study the credit histories of the separate groups.

In [21]:
# create the columns purpose_id and purpose_cat_id in the table purposes
purposes['purpose_id'] = purposes.index

def set_purpose_cat_id(purpose_category):
    """
    input: purpose_category (object) - string that describes the category of the loan purpose
    output: (int64) - integer, identifier of the category of the loan purpose
    The function takes the category of a loan purpose and maps it to a unique
    integer (identifier):
    
    Category:             ID:
    wedding               0
    repair                1
    car                   2
    education             3
    housing/real estate   4
    other                 5
    """
    
    if purpose_category == 'wedding':
        return 0
    elif purpose_category == 'repair':
        return 1
    elif purpose_category == 'car':
        return 2
    elif purpose_category == 'education':
        return 3
    elif purpose_category == 'housing/real estate':
        return 4
    else:
        return 5

# apply the function set_purpose_cat_id to all 'purpose_category' values
# in the table 'purposes' with apply
purposes['purpose_cat_id'] = purposes['purpose_category'].apply(set_purpose_cat_id)
display(purposes.head(10))

,purpose,purpose_category,purpose_id,purpose_cat_id
0,покупка жилья,housing/real estate,0,4
1,приобретение автомобиля,car,1,2
2,дополнительное образование,education,2,3
3,сыграть свадьбу,wedding,3,0
4,операции с жильем,housing/real estate,4,4
5,образование,education,5,3
6,на проведение свадьбы,wedding,6,0
7,покупка жилья для семьи,housing/real estate,7,4
8,покупка недвижимости,housing/real estate,8,4
9,покупка коммерческой недвижимости,housing/real estate,9,4


In [22]:
# add 'purpose_id' and 'purpose_cat_id' to the main table with merge
# and remove the columns 'purpose' and 'purpose_category' from it with drop().
customer_solvency = customer_solvency.merge(purposes, on=['purpose', 'purpose_category'], how='left')
customer_solvency = customer_solvency.drop(['purpose', 'purpose_category'], axis = 1)
display(customer_solvency.head(10))

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose_id,purpose_cat_id
0,1,-8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,0,4
1,1,-4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,1,2
2,0,-5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,0,4
3,3,-4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,2,3
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,3,0
5,0,-926,27,высшее,0,гражданский брак,1,M,компаньон,0,255763,0,4
6,0,-2879,43,высшее,0,женат / замужем,0,F,компаньон,0,240525,4,4
7,0,-152,50,среднее,1,женат / замужем,0,M,сотрудник,0,135823,5,3
8,2,-6929,35,высшее,0,гражданский брак,1,F,сотрудник,0,95856,6,0
9,0,-2188,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425,7,4


In [23]:
# create dictionaries for the education level and the family status and remove the columns from the main table.
# select the columns 'education', 'education_id', remove the duplicates and reset the indices.
# save the result in the table education_dict
education_dict = customer_solvency[['education', 'education_id']].drop_duplicates().reset_index(drop=True)
display(education_dict)

,education,education_id
0,высшее,0
1,среднее,1
2,неоконченное высшее,2
3,начальное,3
4,ученая степень,4


In [24]:
# create the dictionary for the family status in the same way
family_status_dict = customer_solvency[['family_status', 'family_status_id']].drop_duplicates().reset_index(drop=True)
display(family_status_dict)

,family_status,family_status_id
0,женат / замужем,0
1,гражданский брак,1
2,вдовец / вдова,2
3,в разводе,3
4,не женат / не замужем,4


In [25]:
# The names of the education level and the family status are stored in the dictionaries.
# Now they can be removed from the main table with drop()
customer_solvency = customer_solvency.drop(['education', 'family_status'], axis = 1)
display(customer_solvency.head(10))

,children,days_employed,dob_years,education_id,family_status_id,gender,income_type,debt,total_income,purpose_id,purpose_cat_id
0,1,-8437,42,0,0,F,сотрудник,0,253875,0,4
1,1,-4024,36,1,0,F,сотрудник,0,112080,1,2
2,0,-5623,33,1,0,M,сотрудник,0,145885,0,4
3,3,-4124,32,1,0,M,сотрудник,0,267628,2,3
4,0,340266,53,1,1,F,пенсионер,0,158616,3,0
5,0,-926,27,0,1,M,компаньон,0,255763,0,4
6,0,-2879,43,0,0,F,компаньон,0,240525,4,4
7,0,-152,50,1,0,M,сотрудник,0,135823,5,3
8,2,-6929,35,0,1,F,сотрудник,0,95856,6,0
9,0,-2188,41,1,0,M,сотрудник,0,144425,7,4


In [26]:
# Categorize the number of children in the family.
# create the dictionary of the categories of the number of children children_dict
children_data = {'children_cat_id': [0, 1, 2, 3, 4],
                 'children_cat_name': ['No children', 'One child', 'Two children', 
                                       'Three or more children', 'Status unknown']}
children_columns = ['children_cat_id', 'children_cat_name']
children_dict = pd.DataFrame(data = children_data, columns = children_columns)
display(children_dict)

,children_cat_id,children_cat_name
0,0,No children
1,1,One child
2,2,Two children
3,3,Three or more children
4,4,Status unknown


In [27]:
def set_children_cat_id(children):
    """
    input: children (int64) - integer, number of children in the family
    output: (int64) - integer, identifier of the category of the number of children
    The function maps the number of children in the family to one of the categories:
    
    Children:  ID:   Category:
    0          0     No children
    1          1     One child
    2          2     Two children
    >2         3     Three or more children
    <0         4     Status unknown (value -1 in the column 'children')
    """
    
    if children == 0:
        return 0
    elif children == 1:
        return 1
    elif children == 2:
        return 2
    elif children > 2:
        return 3
    else:
        return 4

# apply the function set_children_cat_id to all 'children' values
# in the table 'customer_solvency' with apply
customer_solvency['children_cat_id'] = customer_solvency['children'].apply(set_children_cat_id)
display(customer_solvency.head(10))

,children,days_employed,dob_years,education_id,family_status_id,gender,income_type,debt,total_income,purpose_id,purpose_cat_id,children_cat_id
0,1,-8437,42,0,0,F,сотрудник,0,253875,0,4,1
1,1,-4024,36,1,0,F,сотрудник,0,112080,1,2,1
2,0,-5623,33,1,0,M,сотрудник,0,145885,0,4,0
3,3,-4124,32,1,0,M,сотрудник,0,267628,2,3,3
4,0,340266,53,1,1,F,пенсионер,0,158616,3,0,0
5,0,-926,27,0,1,M,компаньон,0,255763,0,4,0
6,0,-2879,43,0,0,F,компаньон,0,240525,4,4,0
7,0,-152,50,1,0,M,сотрудник,0,135823,5,3,0
8,2,-6929,35,0,1,F,сотрудник,0,95856,6,0,2
9,0,-2188,41,1,0,M,сотрудник,0,144425,7,4,0


Categorize the income level of the customers. Create a dictionary with the identifiers of the income levels and the descriptions of the categories:
- In the table `customer_solvency` create the column `total_income_cat_name` and write there the income categories created with `qcut()`.
- Then create the dictionary `total_income_dict` with the names and identifiers of the obtained categories.
- Map the names and identifiers of the income categories in the main table with `merge()`.
- Remove the names of the income categories from the main table, leaving only the identifiers (for clarity and to save the space taken by the main table).

In [28]:
# create 6 income categories with the same number of customers in each group with `qcut`.
# The parameter `precision=0` is needed for integer group bounds.
customer_solvency['total_income_cat_name'] = pd.qcut(customer_solvency['total_income'], q=6, precision=0)
# find the unique groups with value_counts() and write them to a separate table
total_income_dict = customer_solvency['total_income_cat_name'].value_counts().reset_index()
# there is no need to store the number of records of each group, remove this data with `drop()`
total_income_dict = total_income_dict.drop('total_income_cat_name', axis=1)
# rename the column with the group names `total_income_cat_name`
total_income_dict.columns = ['total_income_cat_name']
# sort the groups by ascending income with `sort_values` and reset the indices in the table.
total_income_dict = total_income_dict.sort_values('total_income_cat_name').reset_index(drop = True)
# the reset indices can be used as identifiers of the customer income categories.
# write the identifiers to the column `total_income_cat_id`
total_income_dict['total_income_cat_id'] = total_income_dict.index
display(total_income_dict)

,total_income_cat_name,total_income_cat_id
0,"(20666.0, 92141.0]",0
1,"(92141.0, 122618.0]",1
2,"(122618.0, 141139.0]",2
3,"(141139.0, 175340.0]",3
4,"(175340.0, 228859.0]",4
5,"(228859.0, 2265604.0]",5


In [29]:
# map the names and identifiers of the income categories in the main table with merge()
customer_solvency = customer_solvency.merge(total_income_dict, on='total_income_cat_name', how='left')
# remove the names of the income categories from the main table with `drop()`
customer_solvency = customer_solvency.drop('total_income_cat_name', axis=1)
display(customer_solvency.head(10))

,children,days_employed,dob_years,education_id,family_status_id,gender,income_type,debt,total_income,purpose_id,purpose_cat_id,children_cat_id,total_income_cat_id
0,1,-8437,42,0,0,F,сотрудник,0,253875,0,4,1,5
1,1,-4024,36,1,0,F,сотрудник,0,112080,1,2,1,1
2,0,-5623,33,1,0,M,сотрудник,0,145885,0,4,0,3
3,3,-4124,32,1,0,M,сотрудник,0,267628,2,3,3,5
4,0,340266,53,1,1,F,пенсионер,0,158616,3,0,0,3
5,0,-926,27,0,1,M,компаньон,0,255763,0,4,0,5
6,0,-2879,43,0,0,F,компаньон,0,240525,4,4,0,5
7,0,-152,50,1,0,M,сотрудник,0,135823,5,3,0,2
8,2,-6929,35,0,1,F,сотрудник,0,95856,6,0,2,1
9,0,-2188,41,1,0,M,сотрудник,0,144425,7,4,0,3


### Conclusion

The following dictionaries are created during the categorization:

- Loan purposes (dictionary `purposes`)
- Monthly income (dictionary `total_income_dict`)
- Customer education level (dictionary `education_dict`)
- Family status (dictionary `family_status_dict`)
- Number of children (dictionary `children_dict`)

All information about the categories is moved to the dictionaries, and only the category identifiers are left in the main table. These transformations are done to reduce the space taken by the main table and for clarity and convenience.

## Step 3. Data Analysis

### 3.1 Dependence Between Having Children and Repaying a Loan on Time

To answer the question, create a new table `children_debt` with the loan debt data `debt` from `customer_solvency` and the information about the number of children `children_cat_id`, `children_cat_name` from the dictionary `children_dict`.
Then present the results in a pivot table for clarity.
Also calculate the ratio of the number of customers with debts to the total number of customers in each group.

In [30]:
children_debt = customer_solvency.merge(children_dict, on='children_cat_id', how='left')[['children_cat_id', 
                                                                             'children_cat_name', 
                                                                             'debt']]
# get the pivot table with pivot_table(). 
children_debt_pivot = children_debt.pivot_table(index = 'children_cat_name', columns = 'debt', 
                                                values = 'children_cat_id', aggfunc = 'count')
# find in each group the share of customers with debts relative to 
# the total number of customers in this group
children_debt_pivot['rate'] = children_debt_pivot[1] / (children_debt_pivot[0] + children_debt_pivot[1])
display(children_debt_pivot)

debt,0,1,rate
children_cat_name,,,
No children,13027,1063,0.075444
One child,4364,444,0.092346
Status unknown,46,1,0.021277
Three or more children,417,39,0.085526
Two children,1858,194,0.094542


### Conclusion

- The minimum share of customers with debts is in the group without children (7.5%).
- Among the customers with one or two children the values differ little: about 9.5% of the customers have debts.
- Among large families (three or more children) the share of customers with debts is about 8.5%. However, this group is relatively small, and further research is needed to confirm that the result is reliable.
- In general the share of loan debts does not change much with the number of children in the family (within 1-2%).

### 3.2 Dependence Between Family Status and Repaying a Loan on Time

To answer the question, create a new table `family_status_debt` with the loan debt data `debt` from `customer_solvency` and the information about the family status `family_status_id`, `family_status` from the dictionary `family_status_dict`.
Then present the results in a pivot table for clarity.
Also calculate the ratio of the number of customers with debts to the total number of customers in each group.

In [31]:
family_status_debt = customer_solvency.merge(family_status_dict, 
                                on='family_status_id', how='left')[['family_status_id',
                                                                   'family_status',
                                                                   'debt']]
# get the pivot table with pivot_table(). 
family_status_pivot = family_status_debt.pivot_table(index = 'family_status', columns = 'debt',
                                                     values = 'family_status_id', aggfunc = 'count')
# find in each group the share of customers with debts relative to 
# the total number of customers in this group
family_status_pivot['rate'] = family_status_pivot[1] / (family_status_pivot[0] + family_status_pivot[1])
display(family_status_pivot)

debt,0,1,rate
family_status,,,
в разводе,1110,85,0.071130
вдовец / вдова,896,63,0.065693
гражданский брак,3762,388,0.093494
женат / замужем,11408,931,0.075452
не женат / не замужем,2536,274,0.097509


### Conclusion

The family status values are in Russian, as in the dataset: `вдовец / вдова` (widower / widow), `не женат / не замужем` (single), `гражданский брак` (civil partnership), `женат / замужем` (married), `в разводе` (divorced).

- The minimum share of customers with debts is in the group `вдовец / вдова` (6.5%).
- The groups `не женат / не замужем` and `гражданский брак` differ little: about 9.5% of the customers have debts. These are the highest values among all groups.
- The group `вдовец / вдова` has the lowest share of debts, about 6.5%. However, this group is relatively small, and further research is needed to confirm that the result is reliable.
- In the categories `женат / замужем` and `в разводе` the share of debts is about the same, 7-7.5%. However, the group of divorced customers is relatively small, and the data may be unreliable.
- According to the data, married customers have fewer loan debts than single customers or those in a civil partnership. Perhaps married couples plan their budget more carefully and treat their obligations more seriously.
- In general the share of loan debts does not change much with the family status (within 3%).

### 3.3 Dependence Between Income Level and Repaying a Loan on Time

To answer the question, create a new table `total_income_debt` with the loan debt data `debt` from `customer_solvency` and the information about the monthly income `total_income_cat_id` and `total_income_cat_name` from the dictionary `total_income_dict`.
Then present the results in a pivot table for clarity.
Also calculate the ratio of the number of customers with debts to the total number of customers in each group.

In [32]:
total_income_debt = customer_solvency.merge(total_income_dict, 
                                on='total_income_cat_id', how='left')[['total_income_cat_id',
                                                                   'total_income_cat_name',
                                                                   'debt']]
# get the pivot table with pivot_table(). 
total_income_pivot = total_income_debt.pivot_table(index = 'total_income_cat_name', columns = 'debt',
                                                     values = 'total_income_cat_id', aggfunc = 'count')
# find in each group the share of customers with debts relative to 
# the total number of customers in this group
total_income_pivot['rate'] = total_income_pivot[1] / (total_income_pivot[0] + total_income_pivot[1])
display(total_income_pivot.sort_values('rate'))

debt,0,1,rate
total_income_cat_name,,,
"(228859.0, 2265604.0]",3326,250,0.069911
"(175340.0, 228859.0]",2917,252,0.079520
"(20666.0, 92141.0]",3291,285,0.079698
"(92141.0, 122618.0]",3277,298,0.083357
"(141139.0, 175340.0]",3642,339,0.085154
"(122618.0, 141139.0]",3259,317,0.088647


### Conclusion

- Customers with a high income (> 230k) repay their loans on time more often (7% of the customers have debts).
- Customers with lower incomes (up to 90k) and with incomes of 175k-230k have debts in 8% of the cases on average. This may be related to a lack of funds or to problems with budget planning.
- Customers with medium incomes (90k - 175k) have debts in 8-9% of the cases. This may be related to problems with budget planning (large loan amounts and short terms can lead to high monthly payments that the customers cannot always afford).
- In general the data does not differ much with the income level (within 2%).

### 3.4 Influence of the Loan Purpose on Repaying It on Time

To answer the question, create a new table `purposes_debt` with the loan debt data `debt` from `customer_solvency` and the information about the loan purposes `purpose_category` and `purpose_cat_id` from the dictionary `purposes`.
Then present the results in a pivot table for clarity.
Also calculate the ratio of the number of customers with debts to the total number of customers in each group.

In [33]:
purposes_debt = customer_solvency.merge(purposes,
                           on='purpose_cat_id', how='left')[['purpose_cat_id',
                                                             'purpose_category',
                                                             'debt']]
# get the pivot table with pivot_table(). 
purposes_debt_pivot = purposes_debt.pivot_table(index = 'purpose_category', columns = 'debt',
                                                     values = 'purpose_cat_id', aggfunc = 'count')
# find in each group the share of customers with debts relative to 
# the total number of customers in this group
purposes_debt_pivot['rate'] = purposes_debt_pivot[1] / (purposes_debt_pivot[0] + purposes_debt_pivot[1])
display(purposes_debt_pivot.sort_values('rate'))

debt,0,1,rate
purpose_category,,,
repair,572,35,0.057661
housing/real estate,151296,11952,0.073214
wedding,6414,558,0.080034
education,32787,3330,0.092200
car,35127,3627,0.093590


### Conclusion

- Loans for apartment repair are overdue in 5.8% of the cases on average. However, the sample in this category is relatively small, so it is hard to judge how reliable the result is.
- Most loans are taken for the purchase of and other operations with real estate. Debts occur for 7.3% of the customers.
- Loans for education and for buying a car are overdue in 9-9.5% of the cases (about 2 percentage points more than the real estate loans). At this stage it is hard to determine the reasons for this difference.
- In general the data on debts does not change much with the purpose of the loan (within about 3.6 percentage points).

## Step 4. Overall Conclusion

1) The study analyzed the statistics on the solvency of bank customers.
Missing values were found in the same records of the columns `days_employed` and `total_income`. Possible reason: a system failure when the data was loaded or exported from the database.
Negative and very high values were also found in `days_employed`. They may have been caused by data entry errors and by failures in the work with the database. It is hard to name the exact reasons for the incorrect data; further investigation with the bank employees is needed.
Since `days_employed` is not used in the further study, its missing values were replaced with zeros for clarity.
The values in `total_income` were replaced with the median values of the groups with the same customer education level.
46 records with a negative number of children were also found. It was assumed that the number of children of these customers is unknown. The data is stored in a separate group.

2) After the conversion of the string data 71 duplicated rows were found. Since the data has no unique customer identifiers, it is impossible to say for sure whether the data of the same customers is duplicated or the information about different customers coincided by chance.
The duplicated records were removed for educational purposes.

3) Among the loan purposes 5 main groups were selected: `wedding`, `repair`, `car`, `education`, `housing/real estate`. The purposes were lemmatized with the library `pymorphy3`.

4) Dictionaries with the following information were created during the categorization:
    - Loan purposes (dictionary `purposes`)
    - Monthly income (dictionary `total_income_dict`)
    - Customer education level (dictionary `education_dict`)
    - Family status (dictionary `family_status_dict`)
    - Number of children (dictionary `children_dict`)

   All information about the categories was moved to the dictionaries, and only the category identifiers were left in the main table. These transformations were done to reduce the space taken by the main table and for clarity and convenience.

5) The loan debts of the customers were analyzed depending on the number of children in the family, the family status, the monthly income level and the loan purpose.
The data is presented in pivot tables. The shares of customers with debts are additionally calculated for each category.
In some categories the samples turned out to be relatively small, which does not allow drawing unambiguous conclusions. A possible solution is to accumulate more data.
In all categories the share of customers with debts did not exceed 10%, and the differences between the separate groups did not exceed about 3.6 percentage points (not counting the 46 customers with an unknown number of children).